# LinguoMT — AfricaS2T Experiment Runner

Run any of the 4 experiment scripts in **debug** or **full** mode.

| Experiment | Model | Dataset | Languages |
|---|---|---|---|
| `FLEURS__SeamlessM4Tv2` | SeamlessM4T-v2-large | FLEURS | Igbo, Yoruba, Swahili |
| `FLEURS__WhisperNLLB` | Whisper-large-v3 + NLLB-distilled-600M | FLEURS | Yoruba, Swahili, Hausa |
| `AfricanCeltic__SeamlessM4Tv2` | SeamlessM4T-v2-large | African-Celtic | Igbo, Yoruba |
| `AfricanCeltic__WhisperNLLB` | Whisper-large-v3 + NLLB-distilled-600M | African-Celtic | Yoruba, Hausa |

**Before running:** set the runtime to **GPU** → Runtime → Change runtime type → T4 GPU (or A100 if available).

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone / Update Repository

In [ ]:
import os, subprocess

REPO_DIR = "/content/LinguoMT-AfricaS2T"
REPO_URL = "https://github.com/prsisda/LinguoMT-AfricaS2T.git"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print("Repo reset to origin/main")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Select Experiment and Mode

In [ ]:
import re, pathlib

# ── EDIT THESE TWO LINES ────────────────────────────────────────────────
EXPERIMENT = "FLEURS__SeamlessM4Tv2"   # options: see table above
DEBUG_MODE = True                       # True ≈ 10 min GPU | False ≈ 30-60 min GPU
# ────────────────────────────────────────────────────────────────────────

VALID_EXPERIMENTS = [
    "FLEURS__SeamlessM4Tv2",
    "FLEURS__WhisperNLLB",
    "AfricanCeltic__SeamlessM4Tv2",
    "AfricanCeltic__WhisperNLLB",
]
assert EXPERIMENT in VALID_EXPERIMENTS, f"EXPERIMENT must be one of: {VALID_EXPERIMENTS}"

script_path = pathlib.Path(f"{EXPERIMENT}/notebooks/run_experiment.py")
assert script_path.exists(), f"Script not found: {script_path}"

mode_str = "True" if DEBUG_MODE else "False"
patched = re.sub(
    r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",
    rf"\g<1>{mode_str}",
    script_path.read_text()
)
script_path.write_text(patched)

print(f"Experiment : {EXPERIMENT}")
print(f"Mode       : {'DEBUG  (fast test)' if DEBUG_MODE else 'FULL   (paper run)'}")
print(f"Script     : {script_path}")

## Step 4 — Run Experiment

In [ ]:
import subprocess, re, pathlib

# Pull latest fixes before running
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("Repository updated to latest.")

# Re-apply mode patch (EXPERIMENT, DEBUG_MODE, script_path from Step 3)
_mode_str = "True" if DEBUG_MODE else "False"
_patched = re.sub(
    r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",
    rf"\g<1>{_mode_str}",
    script_path.read_text()
)
script_path.write_text(_patched)
print(f"Mode re-applied: {'DEBUG' if DEBUG_MODE else 'FULL'}")

In [ ]:
script = str(script_path)
!python "$script"

---
## (Optional) Run All 4 Experiments Sequentially

Set `DEBUG_MODE_ALL` and run this cell to execute all 4 scripts back-to-back.
Results for each experiment are saved before the next one starts.

| Mode | Expected GPU time |
|------|------------------|
| DEBUG | ~40 min total (4 × ~10 min) |
| FULL  | ~2–3 hrs total (4 × 30–60 min) |

In [ ]:
import re, pathlib, subprocess, sys

# ── SELECT PAPER (uncomment the one you want to run) ─────────────────────
PAPER_MODE = "benchmark"    # Paper 1 — zero-shot baselines          ← ACTIVE
# PAPER_MODE = "adaptation" # Paper 2 — fine-tuning comparison
# PAPER_MODE = "audio"      # Paper 3 — audio strategy analysis
# PAPER_MODE = "cascade"    # Paper 4 — cascade vs end-to-end
# PAPER_MODE = "transfer"   # Paper 5 — cross-lingual transfer
# ─────────────────────────────────────────────────────────────────────────

# ── SELECT MODE ──────────────────────────────────────────────────────────
DEBUG_MODE_ALL = True   # True ≈ 40 min total | False ≈ 2–4 hours total
# ─────────────────────────────────────────────────────────────────────────

ALL_EXPERIMENTS = [
    "AfricanCeltic__SeamlessM4Tv2",
    "AfricanCeltic__WhisperNLLB",
    "FLEURS__SeamlessM4Tv2",
    "FLEURS__WhisperNLLB",
]

# Pull latest code first
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("Repository updated to latest.\n")

mode_str  = "True" if DEBUG_MODE_ALL else "False"

for exp in ALL_EXPERIMENTS:
    sp = pathlib.Path(f"{exp}/notebooks/run_experiment.py")
    src = sp.read_text()
    # Patch DEBUG_MODE
    src = re.sub(r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)", rf"\g<1>{mode_str}", src)
    # Patch PAPER_MODE
    src = re.sub(r'(?m)^(PAPER_MODE\s*=\s*)["\']\w+["\']', rf'\g<1>"{PAPER_MODE}"', src)
    sp.write_text(src)

    print(f"\n{'='*60}")
    print(f"  Running : {exp}")
    print(f"  Paper   : {PAPER_MODE}  |  Mode: {'DEBUG' if DEBUG_MODE_ALL else 'FULL'}")
    print(f"{'='*60}\n")

    result = subprocess.run([sys.executable, str(sp)])
    if result.returncode != 0:
        print(f"ERROR in {exp} (exit code {result.returncode}) — continuing...")

print("\nAll experiments finished.")

---
## (Optional) Save Results to Google Drive

Run this cell after your experiment(s) finish to collect all results into a single
timestamped folder in `MyDrive/LinguoMT-AfricaS2T/`.

Folder name: `LinguoMT-AfricaS2T-DEBUG-<TIMESTAMP>` (debug) or `LinguoMT-AfricaS2T-<TIMESTAMP>` (full).

In [ ]:
import shutil, glob
from datetime import datetime
from pathlib import Path

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_base = Path("/content/drive/MyDrive/LinguoMT-AfricaS2T")
drive_base.mkdir(parents=True, exist_ok=True)

# Find all Results_* folders written by the scripts in /content/
result_dirs = sorted(glob.glob("/content/Results_*"))

if not result_dirs:
    print("No Results_* folders found in /content/ — run experiments first.")
else:
    for src in result_dirs:
        results_name = Path(src).name          # e.g. Results_FLEURS__SeamlessM4Tv2_Large_DEBUG
        folder_name  = f"{results_name}-{timestamp}"
        drive_dest   = drive_base / folder_name
        if drive_dest.exists():
            shutil.rmtree(str(drive_dest))
        shutil.copytree(src, str(drive_dest))
        print(f"Saved: {folder_name}")
    print(f"\nAll results saved to:\n  {drive_base}")